# SemanticX Framework Demo

This notebook demonstrates the core features of the SemanticX Framework:
- Agent initialization and configuration
- Conversation state management
- LLM interaction with mock responses
- Tool registration and execution

## Setup

Make sure you're in the SemanticX project root directory and have installed dependencies:
```bash
pip install -r requirements.txt
pip install -r requirements-dev.txt
```

In [ ]:
# Import core SemanticX components
import sys
import asyncio
from typing import Dict, Any, List

# SemanticX imports
from core.base_agent import BaseAgent
from models.state import ConversationState
from services.llm_service import LLMService
from services.tool_handler import ToolHandler

print("✓ SemanticX components imported successfully")
print(f"  - BaseAgent: {BaseAgent}")
print(f"  - ConversationState: {ConversationState}")
print(f"  - LLMService: {LLMService}")
print(f"  - ToolHandler: {ToolHandler}")

## 1. Create a Custom Agent

Agents in SemanticX are created by extending `BaseAgent` and implementing three required methods:
- `load_prompt()` - System prompt for the agent
- `load_tools()` - Available tools for the agent
- `process()` - Main logic to process requests

In [ ]:
class DemoAgent(BaseAgent):
    """Demo agent for the SemanticX framework tutorial."""
    
    def load_prompt(self) -> str:
        return """You are a helpful demo agent for the SemanticX Framework.
        
Your role is to:
1. Understand user requests
2. Respond helpfully and clearly
3. Demonstrate framework capabilities

Be friendly and informative."""
    
    def load_tools(self) -> List[Dict[str, Any]]:
        # This demo doesn't use tools, but you can add them here
        return []
    
    async def process(self, state: ConversationState) -> ConversationState:
        # Call the LLM with the conversation state
        return await self._call_llm_only(state)

print("✓ DemoAgent class created")
print(f"  - Agent name: {DemoAgent.__name__}")
print(f"  - Base class: {DemoAgent.__bases__}")

## 2. Initialize Agent and Conversation State

In [ ]:
# Create agent instance
session_id = "demo-session-001"
agent = DemoAgent(session_id=session_id)

print("\n=== Agent Initialized ===")
print(f"✓ Agent type: {type(agent).__name__}")
print(f"  - Session ID: {agent.session_id}")
print(f"  - Domain: {agent.domain}")
print(f"  - LLM Service: {agent.llm_service.provider}")
print(f"  - LLM Model: {agent.llm_service.model}")

# Create conversation state
state = ConversationState(session_id=session_id)

print("\n=== Conversation State Initialized ===")
print(f"✓ Conversation ID: {state.conversation_id}")
print(f"  - Session ID: {state.session_id}")
print(f"  - Initial intent: {state.intent}")
print(f"  - Initial status: {state.status}")
print(f"  - Auth status: {state.auth_status}")
print(f"  - Message history length: {len(state.conversation_history)}")

## 3. Interact with the Agent

Add a user message and call the agent to process it.

In [ ]:
# Add user message
user_input = "Hello! Can you tell me about the SemanticX Framework?"
state.add_user_message(user_input)

print("\n=== User Message Added ===")
print(f"✓ User said: '{user_input}'")
print(f"  - Message count in history: {len(state.conversation_history)}")
print(f"  - Last message role: {state.conversation_history[-1]['role']}")
print(f"  - Last message content: {state.conversation_history[-1]['content'][:60]}...")

## 4. Call the LLM Service Directly

Demonstrate the LLM service (using mock fallback if no API key is set).

In [ ]:
# Demonstrate LLM service
llm = LLMService()

print("\n=== LLM Service Configuration ===")
print(f"✓ Provider: {llm.provider}")
print(f"  - Model: {llm.model}")
print(f"  - Temperature: {llm.temperature}")
print(f"  - Max tokens: {llm.max_tokens}")
print(f"  - Client initialized: {llm._client is not None}")

# Call LLM synchronously for demo
import asyncio

async def demo_llm_call():
    messages = [
        {"role": "system", "content": "You are helpful."},
        {"role": "user", "content": "Hello!"}
    ]
    response = await llm.generate_completion(messages=messages)
    return response

# Run async function
response = await demo_llm_call()

print("\n=== LLM Response ===")
print(f"✓ Response type: {type(response)}")
print(f"  - Keys: {list(response.keys())}")
print(f"  - Content length: {len(response.get('content', ''))} chars")
print(f"\n  Content preview:")
print(f"  '{response.get('content', 'N/A')}'")

## 5. Process with Agent

Call the agent's process method to get a full response through the LLM.

In [ ]:
# Process with agent
async def agent_interaction():
    print("\n=== Agent Processing Request ===")
    print(f"→ Processing user input...")
    
    # Call agent.process (or agent._call_llm_only for demo)
    updated_state = await agent._call_llm_only(state)
    
    return updated_state

# Run the interaction
updated_state = await agent_interaction()

print("\n=== Updated Conversation State ===")
print(f"✓ Processing complete")
print(f"  - Total messages: {len(updated_state.conversation_history)}")
print(f"  - Status: {updated_state.status}")
print(f"  - Last activity: {updated_state.last_activity}")

# Print conversation
print(f"\n=== Full Conversation ===")
for i, msg in enumerate(updated_state.conversation_history):
    role = msg.get('role', 'unknown').upper()
    content = msg.get('content', '(empty)')
    preview = content[:100] + '...' if len(content) > 100 else content
    print(f"\n[{i}] {role}")
    print(f"    {preview}")

## 6. Tool Registration and Execution

Demonstrate the tool handler for registering and executing custom tools.

In [ ]:
# Define a simple tool function
def calculate_sum(a: int, b: int, user_state=None) -> Dict[str, Any]:
    """Simple tool that adds two numbers."""
    return {"result": a + b, "operation": "sum"}

# Initialize tool handler
tool_handler = ToolHandler()

print("\n=== Tool Handler Initialized ===")
print(f"✓ ToolHandler created")
print(f"  - Registered tools: {tool_handler.list_tools()}")

# Register tool
tool_handler.register_tool(
    "sum_calculator",
    calculate_sum,
    metadata={
        "description": "Adds two integers and returns the result",
        "category": "math"
    }
)

print("\n=== Tool Registered ===")
print(f"✓ Tool 'sum_calculator' registered")
print(f"  - Available tools: {tool_handler.list_tools()}")
print(f"  - Tool info: {tool_handler.get_tool_info('sum_calculator')}")

## 7. Execute Tool

In [ ]:
# Execute the tool
async def execute_tool_demo():
    print("\n=== Tool Execution ===")
    print("→ Executing sum_calculator with a=5, b=3...")
    
    result = await tool_handler.execute_tool(
        "sum_calculator",
        {"a": 5, "b": 3}
    )
    
    return result

tool_result = await execute_tool_demo()

print("\n=== Tool Result ===")
print(f"✓ Execution successful: {tool_result.get('success')}")
print(f"  - Tool name: {tool_result.get('tool_name')}")
print(f"  - Result: {tool_result.get('result')}")
print(f"  - Execution time: {tool_result.get('execution_time')}")

## 8. Multi-Turn Conversation

Continue the conversation with additional turns.

In [ ]:
# Add another user message for multi-turn demo
second_input = "That sounds great! How do I create my own agent?"
updated_state.add_user_message(second_input)

print("\n=== Second User Input ===")
print(f"✓ User said: '{second_input}'")
print(f"  - Total turns: {len([m for m in updated_state.conversation_history if m['role'] == 'user'])}")

# Get agent response
async def second_turn():
    return await agent._call_llm_only(updated_state)

final_state = await second_turn()

print("\n=== Multi-Turn Conversation Complete ===")
print(f"✓ Total messages: {len(final_state.conversation_history)}")
print(f"  - User turns: {len([m for m in final_state.conversation_history if m['role'] == 'user'])}")
print(f"  - Assistant turns: {len([m for m in final_state.conversation_history if m['role'] == 'assistant'])}")

print(f"\n=== Full Conversation History ===")
for i, msg in enumerate(final_state.conversation_history):
    role = msg.get('role', 'unknown').upper()
    content = msg.get('content', '(empty)')
    preview = content[:80] + '...' if len(content) > 80 else content
    print(f"[{i}] {role}: {preview}")

## Summary

This demo showed the key features of SemanticX:

✅ **Agent Creation**: Extend `BaseAgent` with custom logic  
✅ **Conversation Management**: Track state, history, and context  
✅ **LLM Integration**: Call LLM services with automatic fallback to mock mode  
✅ **Tool System**: Register and execute custom tools  
✅ **Multi-Turn Dialogue**: Support complex multi-turn conversations  

## Next Steps

- Read [CONTRIBUTING.md](../CONTRIBUTING.md) to contribute
- Check [AGENT_DEVELOPMENT.md](../AGENT_DEVELOPMENT.md) for detailed agent guide
- Review [tests/](../tests/) for more examples
- Deploy your agent using the FastAPI server in [main.py](../main.py)